# Real-World Exploration: insitro KinDEL (DDR1)

This notebook analyzes the real KinDEL Parquet dump `data/kindel_ddr1_real.parquet`: public columns (`seq_load`, `seq_target_*`) are mapped to the internal schema via `_normalize_kindel_column_aliases` in `src/importer.py`, then counts are depth-normalized and filtered (`min_total_count=10`). Enrichment uses `BetaBinomialConfig` + `summarize_enrichment` from `src/analyzer.py` (delta-method uncertainty).

- **Section 1**: Data integrity (pool totals and sampled count distributions)
- **Section 2**: Bayesian shrinkage and volcano views (library-wide)
- **Section 3**: Scaffold families from an 8-character `molecule_hash` prefix


In [ ]:
from __future__ import annotations

import gc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import norm

from src.analyzer import (
    BetaBinomialConfig,
    aggregate_enrichment_by_scaffold,
    summarize_enrichment,
)
from src.importer import (
    KinDELImportConfig,
    _normalize_kindel_column_aliases,
    import_kindel_counts,
)

sns.set_context("talk")

# Notebook cwd is typically `notebooks/`; parquet lives under repo `data/`.
DATA_PATH = Path("..") / "data" / "kindel_ddr1_real.parquet"
assert DATA_PATH.exists(), f"Missing {DATA_PATH}."

tbl = pd.read_parquet(DATA_PATH)
tbl = _normalize_kindel_column_aliases(tbl)

cfg = KinDELImportConfig(min_total_count=10)
df = import_kindel_counts(tbl, config=cfg)
del tbl
gc.collect()

df.shape, df.columns[:24].tolist()


In [ ]:
cols = [
    "compound_id",
    cfg.input_col,
    *cfg.selected_replicate_cols,
    cfg.output_input_col,
    cfg.output_selected_col,
]
cols = [c for c in cols if c in df.columns]
df[cols].head()


## Section 1 — Data Integrity

Per-pool totals are cheap at any scale. For **histograms**, we subsample row indices once and melt only that subset so we never build a long-form frame on the full ~1M rows.

In [ ]:
input_col = cfg.input_col
rep_cols = list(cfg.selected_replicate_cols)

rng = np.random.default_rng(7)
n = len(df)
sample_n = min(n, 120_000)
ix = rng.choice(n, size=sample_n, replace=False)

plot_df = df.iloc[ix][[input_col, *rep_cols]]
for c in [input_col, *rep_cols]:
    plot_df[c] = pd.to_numeric(plot_df[c], errors="coerce")

long = plot_df.melt(var_name="pool", value_name="count").dropna()
long["count"] = long["count"].clip(lower=0)

plt.figure(figsize=(10, 5))
sns.histplot(
    data=long,
    x="count",
    hue="pool",
    bins=200,
    element="step",
    stat="density",
    common_norm=False,
    log_scale=(True, True),
)
plt.title(f"Raw count distributions (log-log), n={sample_n:,} / {n:,} rows sampled")
plt.xlabel("count")
plt.ylabel("density")
plt.tight_layout()


In [ ]:
pool_totals = {
    c: float(pd.to_numeric(df[c], errors="coerce").fillna(0).clip(lower=0).sum())
    for c in [input_col, *rep_cols]
}
pd.Series(pool_totals).sort_values(ascending=False).to_frame("total_reads")


## Section 2 — Analytical pipeline (Bayesian enrichment)

`summarize_enrichment` applies the Beta–Binomial enrichment model (digamma mean for `log2_enrichment_mean`). We use **`uncertainty_mode="delta"`** for vectorized credible intervals. The next cell runs inference, then **deletes** the imported count frame so only `enriched` remains for downstream plots (one wide table, not two full copies).

In [ ]:
bayes_cfg = BetaBinomialConfig(uncertainty_mode="delta")
enriched = summarize_enrichment(df, config=bayes_cfg)
del df
gc.collect()

enriched.shape


In [ ]:
### Bayesian shrinkage (raw vs posterior mean)

Smoothed library **raw** log2 fold change vs **Bayesian** `log2_enrichment_mean`. Low-count noise is pulled toward 0 (the “Bayesian shield”). We plot with `hexbin` on full rows without building an extra wide DataFrame—only 1d NumPy arrays.


In [ ]:
pc = 0.5
xi = enriched["input_count"].to_numpy(dtype=float, copy=False)
xs = enriched["selected_count"].to_numpy(dtype=float, copy=False)
total_in = float(xi.sum())
total_sel = float(xs.sum())

raw_log2_fc = (
    np.log2(xs + pc)
    - np.log2(total_sel + 2.0 * pc)
    - (np.log2(xi + pc) - np.log2(total_in + 2.0 * pc))
)
bayes_mean = enriched["log2_enrichment_mean"].to_numpy(dtype=float, copy=False)

fig, ax = plt.subplots(figsize=(6.5, 6))
hb = ax.hexbin(
    raw_log2_fc,
    bayes_mean,
    gridsize=90,
    mincnt=1,
    cmap="viridis",
    bins="log",
)
ax.axline((0, 0), slope=1, color="white", ls="--", lw=1, alpha=0.7)
ax.set_xlabel("raw log2 fold change (smoothed library fractions)")
ax.set_ylabel("Bayesian log2_enrichment_mean (digamma)")
ax.set_title("Bayesian shrinkage: noise pulled toward 0")
plt.colorbar(hb, ax=ax, label="log10(count)")
fig.tight_layout()


In [ ]:
### Volcano: enrichment vs −log10(uncertainty)

From the delta-method intervals, approximate posterior **σ** for `log2` enrichment as CI width divided by the normal quantile span, then plot `log2_enrichment_mean` vs `−log10(σ)` (higher = sharper).


lo_q, hi_q = bayes_cfg.credible_interval
z_span = float(norm.ppf(hi_q) - norm.ppf(lo_q))
sigma = (
    enriched["log2_enrichment_ci_high"].to_numpy(dtype=float, copy=False)
    - enriched["log2_enrichment_ci_low"].to_numpy(dtype=float, copy=False)
) / z_span
sigma = np.clip(sigma, 1e-12, np.inf)
neglog10_u = -np.log10(sigma)

x = enriched["log2_enrichment_mean"].to_numpy(dtype=float, copy=False)

fig, ax = plt.subplots(figsize=(7, 6))
hb = ax.hexbin(x, neglog10_u, gridsize=90, mincnt=1, cmap="magma", bins="log")
ax.set_xlabel("log2_enrichment_mean")
ax.set_ylabel(r"$-\log_{10}(\sigma)$  (delta-method, from CI width)")
ax.set_title("Volcano-style view")
plt.colorbar(hb, ax=ax, label="log10(count)")
fig.tight_layout()

In [ ]:
## Section 3 — Scaffold discovery (chemical families)

**`scaffold_id`**: first 8 characters of `molecule_hash` (lightweight family key). We aggregate pooled input/selected counts per scaffold with `aggregate_enrichment_by_scaffold` and rank by `scaffold_log2_enrichment`.


In [ ]:
if "molecule_hash" not in enriched.columns:
    raise ValueError("Parquet table is missing molecule_hash; cannot derive scaffold_id.")

# In-place new column on enriched (no second full-table copy)
enriched["scaffold_id"] = enriched["molecule_hash"].astype(str).str.slice(0, 8)

sc = aggregate_enrichment_by_scaffold(enriched, config=bayes_cfg, scaffold_col="scaffold_id")
n_members = (
    enriched.groupby("scaffold_id", sort=False)
    .size()
    .reset_index(name="n_members")
)
sc = sc.merge(n_members, on="scaffold_id", how="left")

top5_families = sc.sort_values("scaffold_log2_enrichment", ascending=False).head(5)
top5_families


In [ ]:
family_order = top5_families["scaffold_id"].tolist()
mask = enriched["scaffold_id"].isin(family_order)
plot_cols = enriched.loc[mask, ["scaffold_id", "log2_enrichment_mean"]]

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=plot_cols,
    x="scaffold_id",
    y="log2_enrichment_mean",
    order=family_order,
)
plt.xticks(rotation=45, ha="right")
plt.title("Top 5 enriched chemical families (compound-level log2_enrichment_mean)")
plt.tight_layout()